# Validating the 2µ2e isolation-plane ABCD background estimate

This notebook checks that the **ABCD data-driven background estimate is valid** for the
`2mu2e` channel, using the QCD, DY and TTJets MC in
`background_hists_QCD_DY_TTJets_abcd_presel_weighted_v2.coffea`.

The ABCD plane is built from two isolation variables of the leading lepton jets:

* **x = µ-LJ isolation** (`mu_lj_iso`)
* **y = eγ-LJ isolation** (`egm_lj_iso`)

split into four regions by two cuts. On top of the plane there are three **event-level cuts**
that are common to every region: the LJ–LJ **ΔΦ ≥ 2**, the **µ-LJ displacement** (PF-muon
pixel hits ≤ 3), and the **mJJ** cut. The high-mass preselection requires mJJ ≥ 150 GeV;
the low-mass selection (mJJ < 150 GeV) is used here as an independent **validation region**.

**For the ABCD method to work, the two plane variables must be statistically independent**
(uncorrelated) within the preselection. If they are, the signal-region yield factorises as

$$A_{\text{pred}} = \frac{B \cdot C}{D},$$

with the regions defined as

| | eγ pass (iso ≤ cut) | eγ fail (iso > cut) |
|---|---|---|
| **µ pass** (iso ≤ cut) | **A** (signal-like) | **C** |
| **µ fail** (iso > cut) | **B** | **D** |

so that A·D = B·C. This notebook demonstrates the independence three complementary ways
(correlation, conditional-shape overlap, boundary-scan closure), shows the explicit closure
A vs B·C/D, repeats everything in the high- and low-mJJ regimes, and checks that the other
event-level cuts do not sculpt the plane.

---
> **Plane boundaries:** `MU_ISO_CUT = 0.1`, `EGM_ISO_CUT = 0.25`. These were read directly off
> the file — the A/B/C/D region histograms have sharp edges at µ-iso = 0.1 and eγ-iso = 0.25, and
> A+B+C+D reproduces the preselection yield exactly. They are used only for the boundary lines and
> the 2-D-plane–derived quadrants; the A/B/C/D **channel** yields don't depend on them. Change the
> two constants in §0 if you re-run with different thresholds.

## 0  Setup

The next two cells contain the analysis and plotting helpers (self-contained — no external modules to import).

In [ ]:
"""Core ABCD-validation helpers for the SIDM 2mu2e isolation-plane study.

These functions operate on `hist.Hist` objects (or anything implementing the same
.values()/.variances()/.axes API + dict indexing by axis name), so the analysis
logic is independent of coffea and can be unit-tested against mock histograms.
"""
import numpy as np
import functools, operator

# ----------------------------------------------------------------------------- config
SAMPLES = ["QCD", "DY", "TTJets"]
CHANNELS = [
    "2mu2e_noLjCount",
    "2mu2e_abcd_presel",            # high-mass preselection (mJJ >= 150), pre iso-split
    "2mu2e_abcd_presel_lowMass",    # low-mass  preselection (mJJ <  150), pre iso-split
    "2mu2e_abcd_A",
    "2mu2e_abcd_B",
    "2mu2e_abcd_C",
    "2mu2e_abcd_D",
]
# ABCD plane boundaries. These were verified directly against the file: the A/B/C/D
# region 2D histograms have sharp edges at mu_lj_iso = 0.1 and egm_lj_iso = 0.25, and
# A+B+C+D reproduces the presel yield exactly.  (Used only for the boundary lines and
# the 2D-plane-derived quadrants; the A/B/C/D *channel* yields are independent of them.)
MU_ISO_CUT  = 0.1    # mu-LJ isolation boundary  (<= pass)
EGM_ISO_CUT = 0.25   # egm-LJ isolation boundary (<= pass)
REGION_CHANNEL = {"A": "2mu2e_abcd_A", "B": "2mu2e_abcd_B",
                  "C": "2mu2e_abcd_C", "D": "2mu2e_abcd_D"}
# A: mu pass & egm pass | B: mu fail & egm pass | C: mu pass & egm fail | D: mu fail & egm fail
ISO_2D = "mu_lj_iso_egm_lj_iso_2D"

# ----------------------------------------------------------------------------- loading
def load_data(path):
    """Load a coffea dump and return the per-sample dict (strips the 'out' wrapper)."""
    from coffea.util import load
    raw = load(path)
    if isinstance(raw, dict) and "out" in raw and all(s not in raw for s in SAMPLES):
        return raw["out"]
    return raw

def get_hist(data, sample, name):
    return data[sample]["hists"][name]

def total_hist(data, name, samples=None):
    """Sum a named hist over all samples -> total background (axes must match)."""
    samples = samples or [s for s in SAMPLES if s in data]
    hs = [data[s]["hists"][name] for s in samples]
    return functools.reduce(operator.add, [h.copy() for h in hs])

def select_channel(H, ch):
    """Return H with the 'channel' category fixed to `ch` (channel axis removed)."""
    return H[{"channel": ch}]

# ----------------------------------------------------------------------------- yields
def _arr(h, flow=True):
    v = np.asarray(h.values(flow=flow), dtype=float)
    try:
        var = h.variances(flow=flow)
        var = None if var is None else np.asarray(var, dtype=float)
    except Exception:
        var = None
    if var is None:
        var = v.copy()              # fall back to Poisson on the (weighted) counts
    return v, var

def channel_yield(H, ch):
    """Total (weighted) yield + stat error of hist H in channel `ch`, incl. flow."""
    h = select_channel(H, ch)
    v, var = _arr(h, flow=True)
    return float(v.sum()), float(np.sqrt(var.sum()))

def region_yields(data, sample=None):
    """Return {region:(N,err)} for A,B,C,D using the per-event 2D iso hist.
    sample=None -> total background."""
    H = total_hist(data, ISO_2D) if sample is None else get_hist(data, sample, ISO_2D)
    return {r: channel_yield(H, ch) for r, ch in REGION_CHANNEL.items()}

def abcd_closure(yields):
    """Given {region:(N,err)} compute predicted A = B*C/D and the closure ratio.
    Returns dict with observed/predicted A, their errors and obs/pred ratio."""
    (A, sA) = yields["A"]; (B, sB) = yields["B"]; (C, sC) = yields["C"]; (D, sD) = yields["D"]
    out = {"A_obs": A, "A_obs_err": sA, "B": B, "C": C, "D": D}
    if B > 0 and C > 0 and D > 0:
        pred = B * C / D
        rel  = np.sqrt((sB / B) ** 2 + (sC / C) ** 2 + (sD / D) ** 2)
        out["A_pred"] = pred
        out["A_pred_err"] = pred * rel
        if pred > 0:
            ratio = A / pred
            # combine obs & pred relative errors for the ratio band
            rel_ratio = np.sqrt((sA / A) ** 2 + rel ** 2) if A > 0 else rel
            out["ratio"] = ratio
            out["ratio_err"] = ratio * rel_ratio
        else:
            out["ratio"] = np.nan; out["ratio_err"] = np.nan
    else:
        out["A_pred"] = np.nan; out["A_pred_err"] = np.nan
        out["ratio"] = np.nan; out["ratio_err"] = np.nan
    return out

# ----------------------------------------------------------------------- 2D plane utils
def plane_values(h2, fold_flow=True):
    """Return (xcenters, ycenters, W, Var) for a channel-selected 2D iso hist.
    If fold_flow, under/overflow are folded into the edge bins so integrals match
    the unbounded channel yields. x = mu-LJ iso, y = egm-LJ iso."""
    xc = np.asarray(h2.axes[0].centers, dtype=float)
    yc = np.asarray(h2.axes[1].centers, dtype=float)
    W, Var = _arr(h2, flow=fold_flow)
    if fold_flow:
        W = _fold(W); Var = _fold(Var)
    return xc, yc, W, Var

def _fold(a):
    """Fold a (n+2, m+2) flow array into (n, m), adding flow into the edge bins."""
    core = a[1:-1, 1:-1].copy()
    core[0, :]  += a[0, 1:-1]      # x underflow row
    core[-1, :] += a[-1, 1:-1]     # x overflow row
    core[:, 0]  += a[1:-1, 0]      # y underflow col
    core[:, -1] += a[1:-1, -1]     # y overflow col
    core[0, 0]   += a[0, 0];   core[0, -1]  += a[0, -1]
    core[-1, 0]  += a[-1, 0];  core[-1, -1] += a[-1, -1]
    return core

def weighted_corr(xc, yc, W):
    """Weighted Pearson correlation of the two plane variables from a 2D count map."""
    X, Y = np.meshgrid(xc, yc, indexing="ij")
    sw = W.sum()
    if sw <= 0:
        return np.nan
    mx = (W * X).sum() / sw
    my = (W * Y).sum() / sw
    cov = (W * (X - mx) * (Y - my)).sum() / sw
    vx  = (W * (X - mx) ** 2).sum() / sw
    vy  = (W * (Y - my) ** 2).sum() / sw
    if vx <= 0 or vy <= 0:
        return np.nan
    return cov / np.sqrt(vx * vy)

def factorization_test(W):
    """Chi2 / dof and Cramer's V comparing the 2D map to the outer product of its
    1D marginals (the independence hypothesis). Small V => factorizes => independent."""
    row = W.sum(axis=1, keepdims=True)
    col = W.sum(axis=0, keepdims=True)
    tot = W.sum()
    if tot <= 0:
        return {"chi2": np.nan, "ndof": 0, "chi2_per_dof": np.nan, "cramers_v": np.nan}
    E = row * col / tot
    m = E > 0
    chi2 = float(((W[m] - E[m]) ** 2 / E[m]).sum())
    nr = int((row > 0).sum()); nc = int((col > 0).sum())
    ndof = max((nr - 1) * (nc - 1), 1)
    k = min(nr, nc)
    cv = np.sqrt(chi2 / (tot * (k - 1))) if k > 1 else np.nan
    return {"chi2": chi2, "ndof": ndof, "chi2_per_dof": chi2 / ndof, "cramers_v": cv}

def marginal(xc, yc, W, axis):
    """Return (centers, counts) of a 1D marginal. axis=0 -> mu iso, axis=1 -> egm iso."""
    if axis == 0:
        return xc, W.sum(axis=1)
    return yc, W.sum(axis=0)

def conditional_slices(xc, yc, W, slice_axis, edges):
    """Conditional distributions of the *other* variable, in bins of slice_axis.
    Returns list of (label, centers, normalized_density). If independent, the
    normalized shapes overlap."""
    out = []
    cond_centers = yc if slice_axis == 0 else xc
    slice_centers = xc if slice_axis == 0 else yc
    for lo, hi in zip(edges[:-1], edges[1:]):
        sel = (slice_centers >= lo) & (slice_centers < hi)
        if slice_axis == 0:
            counts = W[sel, :].sum(axis=0)
        else:
            counts = W[:, sel].sum(axis=1)
        s = counts.sum()
        dens = counts / s if s > 0 else counts
        out.append((f"[{lo:g}, {hi:g})", cond_centers, dens))
    return out

def quad_counts(xc, yc, W, mu_cut=MU_ISO_CUT, egm_cut=EGM_ISO_CUT):
    """Split a folded 2D map into A,B,C,D yields at the given boundaries.
    A: mu<=cut & egm<=cut ; B: mu>cut & egm<=cut ; C: mu<=cut & egm>cut ; D: mu>cut & egm>cut."""
    xm = xc <= mu_cut          # mu pass
    ym = yc <= egm_cut         # egm pass
    A = W[np.ix_( xm,  ym)].sum()
    B = W[np.ix_(~xm,  ym)].sum()
    C = W[np.ix_( xm, ~ym)].sum()
    D = W[np.ix_(~xm, ~ym)].sum()
    return float(A), float(B), float(C), float(D)

def closure_ratio_from_quads(A, B, C, D):
    """pred/obs = (B*C/D)/A ; ==1 if the two variables are independent."""
    if A > 0 and D > 0:
        return (B * C / D) / A
    return np.nan

def boundary_scan_1d(xc, yc, W, axis, cuts, other_cut):
    """Closure ratio (pred/obs) vs a moving cut on `axis`, other axis fixed."""
    out = []
    for c in cuts:
        if axis == 0:
            A, B, C, D = quad_counts(xc, yc, W, mu_cut=c, egm_cut=other_cut)
        else:
            A, B, C, D = quad_counts(xc, yc, W, mu_cut=other_cut, egm_cut=c)
        out.append(closure_ratio_from_quads(A, B, C, D))
    return np.array(out, dtype=float)

def boundary_scan_2d(xc, yc, W, mu_cuts, egm_cuts):
    """2D grid of closure ratio (pred/obs) over candidate (mu_cut, egm_cut)."""
    R = np.full((len(mu_cuts), len(egm_cuts)), np.nan)
    for i, mc in enumerate(mu_cuts):
        for j, ec in enumerate(egm_cuts):
            A, B, C, D = quad_counts(xc, yc, W, mu_cut=mc, egm_cut=ec)
            R[i, j] = closure_ratio_from_quads(A, B, C, D)
    return R

# ----------------------------------------------------------------- 1D shape comparison
def hist1d(H, ch):
    """Return (edges, centers, counts, errors) for a channel-selected 1D hist."""
    h = select_channel(H, ch)
    edges = np.asarray(h.axes[0].edges, dtype=float)
    centers = np.asarray(h.axes[0].centers, dtype=float)
    v, var = _arr(h, flow=False)
    return edges, centers, np.asarray(v, float), np.sqrt(np.asarray(var, float))

def normalize(counts, edges):
    w = np.diff(edges)
    area = (counts * w).sum()
    return counts / area if area > 0 else counts

In [ ]:
"""Matplotlib plotting helpers for the ABCD-validation notebook.
Pure matplotlib + numpy; mplhep styling is applied separately and optionally."""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

REGION_COLOR = {"A": "#2ca02c", "B": "#1f77b4", "C": "#ff7f0e", "D": "#d62728"}

def plot_2d_plane(xc, yc, W, ax=None, title="", mu_cut=0.1, egm_cut=0.2, cmap="viridis"):
    """Filled 2D iso-iso plane (log color) with ABCD boundary lines + region labels."""
    if ax is None:
        _, ax = plt.subplots(figsize=(5.4, 4.6))
    ex = _edges_from_centers(xc); ey = _edges_from_centers(yc)
    Wm = np.ma.masked_where(W <= 0, W)
    vmin = max(W[W > 0].min(), 1e-6) if np.any(W > 0) else 1e-3
    pc = ax.pcolormesh(ex, ey, Wm.T, norm=LogNorm(vmin=vmin, vmax=W.max() if W.max() > 0 else 1),
                       cmap=cmap, shading="auto")
    plt.colorbar(pc, ax=ax, label="Weighted events")
    ax.axvline(mu_cut, color="white", lw=1.4, ls="--")
    ax.axhline(egm_cut, color="white", lw=1.4, ls="--")
    xhi = ex[-1]; yhi = ey[-1]
    pos = {"A": (mu_cut/2, egm_cut/2),
           "B": ((mu_cut+xhi)/2, egm_cut/2),
           "C": (mu_cut/2, (egm_cut+yhi)/2),
           "D": ((mu_cut+xhi)/2, (egm_cut+yhi)/2)}
    for r, (px, py) in pos.items():
        ax.text(px, py, r, color="white", fontsize=15, fontweight="bold",
                ha="center", va="center",
                bbox=dict(boxstyle="round,pad=0.18", fc="black", alpha=0.45, ec="none"))
    ax.set_xlabel(r"$\mu$-LJ isolation"); ax.set_ylabel(r"$e\gamma$-LJ isolation")
    ax.set_title(title)
    return ax

def plot_closure_bars(labels, A_obs, A_obs_err, A_pred, A_pred_err, ax=None):
    """Grouped bars: observed A vs predicted B*C/D, per sample/total."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6.2, 4.2))
    x = np.arange(len(labels)); w = 0.38
    ax.bar(x - w/2, A_obs, w, yerr=A_obs_err, capsize=3, label="Observed A", color="#2ca02c")
    ax.bar(x + w/2, A_pred, w, yerr=A_pred_err, capsize=3, label=r"Predicted $B\cdot C/D$",
           color="#9467bd")
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("Weighted events in A"); ax.legend()
    ax.set_title("ABCD closure: observed vs predicted signal-region yield")
    return ax

def plot_ratio_points(labels, ratio, ratio_err, ax=None):
    """Closure ratio (obs/pred) per sample/total with a band at 1."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6.2, 3.4))
    x = np.arange(len(labels))
    ax.axhspan(0.8, 1.2, color="0.85", label="+/-20%")
    ax.axhline(1.0, color="k", lw=1)
    ax.errorbar(x, ratio, yerr=ratio_err, fmt="o", color="#d62728", capsize=4)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("A(obs) / A(pred)"); ax.legend()
    ax.set_title("ABCD closure ratio")
    return ax

def plot_conditionals(slices, ax=None, xlabel="", title="", logy=False):
    """Overlay normalized conditional distributions (list of (label, centers, dens))."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6.0, 4.2))
    for lab, c, d in slices:
        ax.step(c, d, where="mid", label=lab)
    ax.set_xlabel(xlabel); ax.set_ylabel("a.u. (norm.)")
    if logy:
        ax.set_yscale("log")
    ax.legend(title="slice", fontsize=8); ax.set_title(title)
    return ax

def plot_shape_compare(curves, ax=None, xlabel="", title="", logy=False, ratio_ref=0):
    """curves: list of (label, edges, counts, errors). Overlays normalized shapes
    with a ratio sub-panel vs curves[ratio_ref]. Returns (ax, ax_ratio)."""
    if ax is None:
        fig = plt.figure(figsize=(6.0, 5.0))
        gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.07)
        ax = fig.add_subplot(gs[0]); axr = fig.add_subplot(gs[1], sharex=ax)
    else:
        axr = None
    norm = []
    for lab, edges, counts, errs in curves:
        w = np.diff(edges); area = (counts * w).sum()
        n = counts / area if area > 0 else counts
        e = errs / area if area > 0 else errs
        cen = 0.5 * (edges[:-1] + edges[1:])
        ax.step(cen, n, where="mid", label=lab)
        ax.errorbar(cen, n, yerr=e, fmt="none", alpha=0.4)
        norm.append((cen, n, e))
    ax.set_ylabel("a.u. (norm.)"); ax.legend(); ax.set_title(title)
    if logy:
        ax.set_yscale("log")
    if axr is not None:
        cen0, n0, e0 = norm[ratio_ref]
        for k, (lab, edges, counts, errs) in enumerate(curves):
            cen, n, e = norm[k]
            with np.errstate(divide="ignore", invalid="ignore"):
                r = np.where(n0 > 0, n / n0, np.nan)
            axr.step(cen, r, where="mid")
        axr.axhline(1, color="k", lw=0.8)
        axr.set_ylim(0.4, 1.6); axr.set_ylabel(f"/ {curves[ratio_ref][0]}")
        axr.set_xlabel(xlabel)
        plt.setp(ax.get_xticklabels(), visible=False)
    else:
        ax.set_xlabel(xlabel)
    return ax, axr

def plot_scan_1d(cuts, ratios, ax=None, xlabel="", default_cut=None, title=""):
    if ax is None:
        _, ax = plt.subplots(figsize=(5.8, 3.8))
    ax.axhspan(0.8, 1.2, color="0.88")
    ax.axhline(1, color="k", lw=1)
    ax.plot(cuts, ratios, "o-", color="#1f77b4", ms=4)
    if default_cut is not None:
        ax.axvline(default_cut, color="#d62728", ls="--", label=f"nominal = {default_cut:g}")
        ax.legend()
    ax.set_xlabel(xlabel); ax.set_ylabel(r"$(B\cdot C/D)\,/\,A$"); ax.set_title(title)
    return ax

def plot_scan_2d(mu_cuts, egm_cuts, R, ax=None, mu_nom=0.1, egm_nom=0.2):
    if ax is None:
        _, ax = plt.subplots(figsize=(5.6, 4.6))
    em = _edges_from_centers(np.asarray(mu_cuts)); ee = _edges_from_centers(np.asarray(egm_cuts))
    pc = ax.pcolormesh(em, ee, R.T, cmap="RdBu_r", vmin=0.5, vmax=1.5, shading="auto")
    plt.colorbar(pc, ax=ax, label=r"$(B\cdot C/D)/A$")
    ax.plot(mu_nom, egm_nom, "k*", ms=14)
    ax.set_xlabel(r"$\mu$-LJ iso cut"); ax.set_ylabel(r"$e\gamma$-LJ iso cut")
    ax.set_title("Closure ratio vs choice of plane boundaries")
    return ax

def _edges_from_centers(c):
    c = np.asarray(c, dtype=float)
    if len(c) == 1:
        return np.array([c[0]-0.5, c[0]+0.5])
    mid = 0.5 * (c[:-1] + c[1:])
    first = c[0] - (mid[0] - c[0]); last = c[-1] + (c[-1] - mid[-1])
    return np.concatenate([[first], mid, [last]])

In [ ]:
# notebook imports + CMS style
import numpy as np
import matplotlib.pyplot as plt
try:
    import mplhep as hep
    plt.style.use(hep.style.CMS)
except Exception as e:
    print("mplhep not available, using default style:", e)
try:
    import pandas as pd
except Exception as e:
    pd = None; print("pandas not available - tables will print as dicts")
try:
    from IPython.display import display
except Exception:
    display = print
plt.rcParams["figure.dpi"] = 110

def show_table(df, fmt=None, caption=None):
    """Render a DataFrame nicely; fall back gracefully if jinja2/Styler is absent."""
    try:
        sty = df.style
        if fmt is not None:
            sty = sty.format(fmt)
        if caption:
            sty = sty.set_caption(caption)
        display(sty); return
    except Exception:
        if caption:
            print("\n" + caption)
        try:
            display(df.round(4))
        except Exception:
            print(df.to_string())

In [ ]:
# ---- locate & load the coffea file -------------------------------------------
import glob, os
# >>> EDIT if the file lives elsewhere <<<
COFFEA_FILE = "background_hists_QCD_DY_TTJets_abcd_presel_weighted_v2.coffea"

def _resolve(fn):
    cands = [fn, os.path.join(".", fn),
             os.path.expanduser(os.path.join("~/Documents/SIDM/SIDM", os.path.basename(fn)))]
    cands += glob.glob("**/" + os.path.basename(fn), recursive=True)
    cands += glob.glob("**/*abcd_presel_weighted*.coffea", recursive=True)
    for c in cands:
        if c and os.path.exists(c):
            return c
    raise FileNotFoundError("Could not find %r - set COFFEA_FILE to the full path." % fn)

path = _resolve(COFFEA_FILE)
print("Loading:", path)
data = load_data(path)
print("Samples :", list(data.keys()))

# sanity: axes + channels actually present
H2 = get_hist(data, [s for s in SAMPLES if s in data][0], ISO_2D)
print("2D hist axes:", [getattr(a, "name", "?") for a in H2.axes])
ch_axis = [a for a in H2.axes if getattr(a, "name", "") == "channel"][0]
print("channels    :", list(ch_axis))
print("plane boundaries: MU_ISO_CUT = %.3g , EGM_ISO_CUT = %.3g" % (MU_ISO_CUT, EGM_ISO_CUT))

## 1  Region yields and the explicit ABCD closure

First the per-region yields (weighted to lumi×σ — this file is already scaled). The A/B/C/D
**channels** were filled with the real iso lambdas, so these are the precise numbers.
As a sanity check, A+B+C+D should reproduce the high-mass preselection yield (the iso split is
exhaustive). Then we compute the closure prediction $A_{\text{pred}}=B\,C/D$ and the ratio
$A_{\text{obs}}/A_{\text{pred}}$ — **a ratio consistent with 1 is the headline ABCD-validity result**.

In [ ]:
# per-channel yields, per sample + total (from the per-event 2D iso hist)
samples = [s for s in SAMPLES if s in data]
rows = []
Htot = total_hist(data, ISO_2D)
for ch in CHANNELS:
    n, _ = channel_yield(Htot, ch)
    row = {"channel": ch, "total": n}
    for s in samples:
        row[s] = channel_yield(get_hist(data, s, ISO_2D), ch)[0]
    rows.append(row)

if pd is not None:
    ydf = pd.DataFrame(rows).set_index("channel")
    show_table(ydf, "{:.2f}", "Weighted yields per channel")
else:
    ydf = None
    for r in rows: print(r)

# partition check: A+B+C+D == high-mass presel
ABCD = sum(channel_yield(Htot, REGION_CHANNEL[r])[0] for r in "ABCD")
presel = channel_yield(Htot, "2mu2e_abcd_presel")[0]
print("\nA+B+C+D = %.2f   |   high-mass presel = %.2f   |   diff = %.2e"
      % (ABCD, presel, ABCD - presel))

In [ ]:
# ABCD closure per sample and for the total background
def _closure_row(label, yields):
    d = abcd_closure(yields); d = {"sample": label, **d}; return d

crows = [_closure_row("Total", region_yields(data))]
for s in samples:
    crows.append(_closure_row(s, region_yields(data, s)))

cols = ["A_obs", "A_obs_err", "B", "C", "D", "A_pred", "A_pred_err", "ratio", "ratio_err"]
if pd is not None:
    cdf = pd.DataFrame(crows).set_index("sample")[cols]
    show_table(cdf, "{:.3g}", "ABCD closure  (ratio = A_obs / A_pred)")
else:
    cdf = None
    for r in crows: print({k: r.get(k) for k in ["sample"] + cols})

print("\nTotal-background closure: A_obs/A_pred = %.3f ± %.3f"
      % (crows[0]["ratio"], crows[0]["ratio_err"]))

In [ ]:
# visualise the closure
labels   = [r["sample"] for r in crows]
A_obs    = np.array([r["A_obs"]     for r in crows])
A_obs_e  = np.array([r["A_obs_err"] for r in crows])
A_pred   = np.array([r["A_pred"]    for r in crows])
A_pred_e = np.array([r["A_pred_err"]for r in crows])
ratio    = np.array([r["ratio"]     for r in crows])
ratio_e  = np.array([r["ratio_err"] for r in crows])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
plot_closure_bars(labels, A_obs, A_obs_e, A_pred, A_pred_e, ax=axes[0])
plot_ratio_points(labels, ratio, ratio_e, ax=axes[1])
plt.tight_layout(); plt.show()

## 2  The isolation–isolation plane

The headline object. Colour = weighted events; dashed white lines are the A/B/C/D boundaries.
**Independence shows up as a plane with no diagonal structure** — the population of one variable
should not drift as you move along the other. We draw it for the high-mass preselection and for
the low-mass validation region, then per sample.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.0))
for ax, ch, title in [(axes[0], "2mu2e_abcd_presel",         "Total bkg - high-mass presel (mJJ >= 150)"),
                      (axes[1], "2mu2e_abcd_presel_lowMass", "Total bkg - low-mass presel (mJJ < 150)")]:
    xc, yc, W, _ = plane_values(select_channel(total_hist(data, ISO_2D), ch))
    plot_2d_plane(xc, yc, W, ax=ax, title=title, mu_cut=MU_ISO_CUT, egm_cut=EGM_ISO_CUT)
plt.tight_layout(); plt.show()

In [ ]:
# per-sample plane (high-mass presel)
fig, axes = plt.subplots(1, len(samples), figsize=(5.4*len(samples), 4.6))
axes = np.atleast_1d(axes)
for ax, s in zip(axes, samples):
    xc, yc, W, _ = plane_values(select_channel(get_hist(data, s, ISO_2D), "2mu2e_abcd_presel"))
    plot_2d_plane(xc, yc, W, ax=ax, title=f"{s} - high-mass presel")
plt.tight_layout(); plt.show()

## 3  Independence of the two isolation cuts

Three quantitative tests, all on the preselection plane (before the iso split):

1. **Weighted Pearson correlation** $r$ between µ-iso and eγ-iso (linear dependence).
2. **Factorisation $\chi^2$ / Cramér's V**: compares the 2-D map to the outer product of its
   1-D marginals — i.e. the independence hypothesis — capturing *any* dependence, not just linear.
   $V\!\to\!0$ means the plane factorises.
3. **Conditional-shape overlap**: the µ-iso distribution measured in slices of eγ-iso (and vice
   versa). **If the variables are independent these normalised shapes lie on top of each other.**

In [ ]:
rows = []
for ch, lab in [("2mu2e_abcd_presel", "high-mJJ"), ("2mu2e_abcd_presel_lowMass", "low-mJJ")]:
    for s in ["Total"] + samples:
        H = total_hist(data, ISO_2D) if s == "Total" else get_hist(data, s, ISO_2D)
        xc, yc, W, _ = plane_values(select_channel(H, ch))
        ft = factorization_test(W)
        rows.append({"region": lab, "sample": s,
                     "pearson_r": weighted_corr(xc, yc, W),
                     "cramers_v": ft["cramers_v"], "chi2/dof": ft["chi2_per_dof"]})
if pd is not None:
    corrdf = pd.DataFrame(rows)
    show_table(corrdf, {"pearson_r": "{:+.3f}", "cramers_v": "{:.3f}", "chi2/dof": "{:.2f}"},
               "Independence metrics  (|r|, V ~ 0  ->  independent)")
else:
    for r in rows: print(r)

In [ ]:
# conditional-shape overlap (total background, high-mass presel)
xc, yc, W, _ = plane_values(select_channel(total_hist(data, ISO_2D), "2mu2e_abcd_presel"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
sl = conditional_slices(xc, yc, W, slice_axis=1, edges=[0, EGM_ISO_CUT, 0.4, 0.8])
plot_conditionals([(f"egm-iso {l}", c, d) for l, c, d in sl], ax=axes[0],
                  xlabel=r"$\mu$-LJ isolation", logy=True,
                  title=r"$\mu$-iso shape in slices of $e\gamma$-iso")
sl2 = conditional_slices(xc, yc, W, slice_axis=0, edges=[0, MU_ISO_CUT, 0.3, 0.8])
plot_conditionals([(f"mu-iso {l}", c, d) for l, c, d in sl2], ax=axes[1],
                  xlabel=r"$e\gamma$-LJ isolation", logy=True,
                  title=r"$e\gamma$-iso shape in slices of $\mu$-iso")
plt.tight_layout(); plt.show()
print("Overlapping normalised shapes  =>  the two isolations are independent.")

### 3b  Boundary-scan closure (robustness)

If the closure only worked at one finely-tuned cut it would be a coincidence. Here we re-evaluate
$(B\,C/D)/A$ from the preselection plane as the boundaries are moved across the plane. **A flat band
at 1 over a wide range of cuts** is strong evidence the factorisation is real. (These come from the
binned 2-D plane, so expect small binning wiggles vs. the exact channel numbers in §1.)

In [ ]:
xc, yc, W, _ = plane_values(select_channel(total_hist(data, ISO_2D), "2mu2e_abcd_presel"))
mu_cuts  = np.round(np.arange(0.04, 0.41, 0.02), 3)
egm_cuts = np.round(np.arange(0.06, 0.61, 0.02), 3)
r_mu  = boundary_scan_1d(xc, yc, W, axis=0, cuts=mu_cuts,  other_cut=EGM_ISO_CUT)
r_egm = boundary_scan_1d(xc, yc, W, axis=1, cuts=egm_cuts, other_cut=MU_ISO_CUT)
R     = boundary_scan_2d(xc, yc, W, mu_cuts, egm_cuts)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
plot_scan_1d(mu_cuts,  r_mu,  ax=axes[0], xlabel=r"$\mu$-LJ iso cut",  default_cut=MU_ISO_CUT,
             title="closure vs mu cut (egm fixed)")
plot_scan_1d(egm_cuts, r_egm, ax=axes[1], xlabel=r"$e\gamma$-LJ iso cut", default_cut=EGM_ISO_CUT,
             title="closure vs egm cut (mu fixed)")
plot_scan_2d(mu_cuts, egm_cuts, R, ax=axes[2], mu_nom=MU_ISO_CUT, egm_nom=EGM_ISO_CUT)
plt.tight_layout(); plt.show()

## 4  High-mJJ vs low-mJJ (validation region)

The mJJ cut splits the preselection into the high-mass region (signal-adjacent) and the low-mass
region (a signal-depleted **validation region**). If the iso plane is independent of mJJ, the
isolation shapes should match between the two regimes, and the closure / correlation should be the
same. Matching here means **the mJJ cut does not sculpt the ABCD plane**, so the low-mass region is a
faithful validation of the high-mass estimate.

In [ ]:
def marg(ch, axis):
    H = select_channel(total_hist(data, ISO_2D), ch)
    xc, yc, W, Var = plane_values(H)
    cen, counts = marginal(xc, yc, W,   axis)
    _,   var    = marginal(xc, yc, Var, axis)
    edges = np.asarray(H.axes[axis].edges, float)
    return edges, counts, np.sqrt(var)

for axis, name in [(0, r"$\mu$-LJ isolation"), (1, r"$e\gamma$-LJ isolation")]:
    eh, ch_, erh = marg("2mu2e_abcd_presel",         axis)
    el, cl_, erl = marg("2mu2e_abcd_presel_lowMass", axis)
    plot_shape_compare([("high-mJJ (>=150)", eh, ch_, erh),
                        ("low-mJJ (<150)",   el, cl_, erl)],
                       xlabel=name, logy=True, title=f"{name}: high- vs low-mJJ shape")
    plt.show()

In [ ]:
# closure + correlation, high vs low mJJ, side by side (total background)
def quad_closure_from_plane(ch):
    xc, yc, W, _ = plane_values(select_channel(total_hist(data, ISO_2D), ch))
    A, B, C, D = quad_counts(xc, yc, W)
    return A, B, C, D, closure_ratio_from_quads(A, B, C, D), weighted_corr(xc, yc, W)

print("Region        A        B        C        D     (BC/D)/A    pearson_r")
for ch, lab in [("2mu2e_abcd_presel", "high-mJJ"), ("2mu2e_abcd_presel_lowMass", "low-mJJ")]:
    A, B, C, D, rr, rho = quad_closure_from_plane(ch)
    print("%-9s %8.1f %8.1f %8.1f %8.1f %10.3f %11.3f" % (lab, A, B, C, D, rr, rho))
print("\nSimilar closure ratio and r in both regimes => mJJ cut is independent of the iso plane.")

## 5  Do the other event-level cuts sculpt the plane?

The ΔΦ and displacement cuts are applied identically in all four regions, but we should check the
*kinematics behind them* don't differ region-to-region. Below, the LJ–LJ separation
($\Delta R$, $|\Delta\eta|$ — the variables behind the ΔΦ cut) and the µ-LJ PF-muon **min pixel
hits** (the displacement variable) are overlaid, normalised, across A/B/C/D. **Region-independent
shapes mean these cuts factorise from the iso plane too**, so they don't bias the ABCD estimate.

In [ ]:
varlist = [("mu_lj_egm_lj_dR",      r"$\Delta R(\mu\mathrm{-LJ}, e\gamma\mathrm{-LJ})$"),
           ("mu_lj_egm_lj_absdeta", r"$|\Delta\eta|(\mu\mathrm{-LJ}, e\gamma\mathrm{-LJ})$"),
           ("pf_mu_lj_pfMuon_min_trkNumPixelHits", "PF-mu min pixel hits (displacement)")]

for hn, xlabel in varlist:
    try:
        Hn = total_hist(data, hn)
    except Exception as e:
        print("skip", hn, e); continue
    curves = []
    for r in "ABCD":
        ch = REGION_CHANNEL[r]
        e, c, v, er = hist1d(Hn, ch)
        if np.nansum(v) > 0:
            curves.append((r, e, v, er))
    if curves:
        plot_shape_compare(curves, xlabel=xlabel, logy=False,
                           title=f"{xlabel}: shape across A/B/C/D")
        plt.show()
    else:
        print("no entries for", hn)

## 6  Summary

Pulls together the key numbers into a verdict.

In [ ]:
tot_close = abcd_closure(region_yields(data))
xc, yc, W, _ = plane_values(select_channel(total_hist(data, ISO_2D), "2mu2e_abcd_presel"))
xl, yl, Wl, _ = plane_values(select_channel(total_hist(data, ISO_2D), "2mu2e_abcd_presel_lowMass"))
r_hi = weighted_corr(xc, yc, W);  v_hi = factorization_test(W)["cramers_v"]
r_lo = weighted_corr(xl, yl, Wl); v_lo = factorization_test(Wl)["cramers_v"]

print("="*64)
print("ABCD VALIDATION SUMMARY  (QCD + DY + TTJets)")
print("="*64)
print("Plane boundaries:            mu_iso <= %.3g , egm_iso <= %.3g" % (MU_ISO_CUT, EGM_ISO_CUT))
print("-"*64)
print("Closure  A_obs/A_pred:       %.3f +/- %.3f" % (tot_close["ratio"], tot_close["ratio_err"]))
print("  A_obs = %.1f   B*C/D = %.1f" % (tot_close["A_obs"], tot_close["A_pred"]))
print("-"*64)
print("Independence (high-mJJ):     pearson r = %+.3f , Cramer's V = %.3f" % (r_hi, v_hi))
print("Independence (low-mJJ):      pearson r = %+.3f , Cramer's V = %.3f" % (r_lo, v_lo))
print("="*64)
ok = (abs(tot_close["ratio"] - 1) < 3*max(tot_close["ratio_err"], 1e-9)) and (abs(r_hi) < 0.15)
print("Verdict:", "consistent with a valid ABCD plane"
      if ok else "inspect the plots — some tension; check stats / boundaries")
print("""
Caveats:
 * Limited MC stats, especially in B/C/D for DY & TTJets, dominate the closure error.
 * Boundary-scan ratios use the binned 2-D plane (0.016 bins), so they differ slightly
   from the exact channel yields in section 1 near the boundary.
 * Plane boundaries (mu_iso <= 0.1, egm_iso <= 0.25) were read off the sharp edges of the
   A/B/C/D region histograms; the section-1 closure uses the channels directly.
""")